# RACE3 test: MACURA vs MBPO with equal updates, 4 seeds on 2 Kaggle accounts

**This session trains seeds 2, 3** (account nouradon: seeds 0, 1; account silvergjeka01: seeds 2, 3; the same
notebook otherwise). The two sessions run at the same time; their outputs are merged afterwards.

**Settings (decided before the run):**
- Task **race3** = race2 with a deadlier chute (up to 50% lift lost instead of 35%); everything else unchanged.
- **50,000 steps** per algorithm (5,000 random warm-up + 45,000 learning), the paper protocol of race2.
- **MBPO: 12 SAC updates per real step** (was 8) = MACURA's average in the race2 pilot (11.7), so neither method
  simply trains more. MACURA keeps its adaptive updates (up to 16).
- Exploration as in the MACURA paper (disclosed): MACURA pink noise, MBPO without noise, SAC samples its policy.
- **MACURA and MBPO of every seed are trained at the same time** (one process each); SAC runs when a slot is free.

**Decision rule (fixed before the run):** race3 becomes the final study if MACURA's **average return while learning**
is higher than MBPO's in **at least 3 of the 4 seeds** and on their average; otherwise the race2 plan stays. Drones
broken, test return, crash rate and the chute check are reported either way. If race3 is chosen, these runs are the
final study (the final notebook analyses them); the race2 pilot is reported next to them in any case.

## 1. Settings

In [ ]:
import os, sys, time, subprocess

SEEDS    = [2, 3]             # <- this session's seeds (nouradon: 0, 1 | silvergjeka01: 2, 3)
TASK     = "race3"
STEPS    = 50000              # per algorithm: 5,000 warm-up + 45,000 learning
TOGETHER = ["macura", "mbpo"] # trained at the same time, for every seed
AFTER    = ["sac"]            # trained when a slot is free
MBPO_UTD = 12                 # MBPO updates per real step (= MACURA's average in the race2 pilot)
MAX_JOBS = None               # trainings at the same time; None = min(4, CPU cores)
REPO, BRANCH = "silvergjeka22/macura-drone", "race3"
ROOT = "/kaggle/working/macura-drone"
OUT  = "/kaggle/working/runs"

T0 = time.time()
DEADLINE = T0 + 11.0 * 3600   # Kaggle stops a session at 12 h and then saves nothing: training stops at 11 h
print("=" * 78)
print(f"  SEEDS {SEEDS}   |   task {TASK}   |   {STEPS:,} steps   |   MBPO {MBPO_UTD} updates per step")
print(f"  trained together: {', '.join(TOGETHER)}   then: {', '.join(AFTER)}")
print("=" * 78)
os.environ.update({"MACURA_TASK": TASK, "MACURA_STEPS": str(STEPS), "MACURA_SEEDS": " ".join(map(str, SEEDS)),
                   "MACURA_ALGOS": " ".join(TOGETHER + AFTER), "MBPO_UTD": str(MBPO_UTD),
                   "MACURA_OUTPUT_ROOT": OUT, "MACURA_DEADLINE": str(DEADLINE)})

token = os.environ.get("GITHUB_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)
sys.path.insert(0, ROOT)
print("code:", subprocess.run(["git", "-C", ROOT, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.bootstrap import setup
setup(root=ROOT)

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display
import src.config.config as cfg
from src.analysis import results as R
from src.viz import figures as F

N_JOBS = MAX_JOBS or min(4, os.cpu_count() or 1)
N_GPU = torch.cuda.device_count()
print(f"config: task {cfg.TASK} | seeds {cfg.SEEDS} | max lift loss {cfg.ENV['vrs_loss']:.0%} | "
      f"MBPO updates {cfg.ROLLOUT['mbpo']['fixed_gradient_steps']} | MACURA xi {cfg.ROLLOUT['macura']['xi']}, "
      f"up to {cfg.ROLLOUT['macura']['gradient_steps_max']} updates | warm-up {cfg.WARMUP_RANDOM_STEPS}")
print(f"machine: {os.cpu_count()} CPU cores, {N_GPU} GPU(s) -> {N_JOBS} trainings at the same time")

CHILD = '''
import sys
sys.path.insert(0, {root!r})
from src.config import config as cfg
from src.training.train import train_algo
train_algo({algo!r}, cfg.CFG, cfg.OUTPUT_ROOT, seeds=[{seed}])
'''


def _clock():
    return f"{(time.time() - T0) / 3600:4.1f} h"


def _last_eval(log):
    try:
        lines = [l for l in open(log).read().splitlines() if l.startswith("[") and " step " in l]
        return lines[-1].strip() if lines else "(warm-up)"
    except OSError:
        return "(no log yet)"


def run_jobs(jobs, n_jobs, report_every=900):
    # each training is its own process (one CPU thread each); a new one starts when a slot is free
    os.makedirs(f"{OUT}/joblogs", exist_ok=True)
    pending, running, last = list(jobs), {}, 0.0
    while pending or running:
        while pending and len(running) < n_jobs:
            algo, seed = pending.pop(0)
            if os.path.exists(f"{OUT}/logs/{algo}_seed{seed}.json"):
                print(f"[{_clock()}] {algo} seed {seed}: already finished, reloaded"); continue
            if time.time() > DEADLINE:
                print(f"[{_clock()}] {algo} seed {seed}: no session time left, skipped"); continue
            env = dict(os.environ, MACURA_SEEDS=str(seed), MACURA_ALGOS=algo, PYTHONUNBUFFERED="1",
                       OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
            if N_GPU > 1:
                env["CUDA_VISIBLE_DEVICES"] = str(len(running) % N_GPU)
            log = f"{OUT}/joblogs/{algo}_seed{seed}.txt"
            p = subprocess.Popen([sys.executable, "-u", "-c", CHILD.format(root=ROOT, algo=algo, seed=seed)],
                                 env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)
            running[(algo, seed)] = (p, log, time.time())
            print(f"[{_clock()}] started {algo} seed {seed}  (log: joblogs/{algo}_seed{seed}.txt)", flush=True)
        for (algo, seed), (p, log, t0) in list(running.items()):
            if p.poll() is None:
                continue
            del running[(algo, seed)]
            mins = (time.time() - t0) / 60
            if p.returncode == 0:
                final = [l for l in open(log).read().splitlines() if "FINAL" in l]
                print(f"[{_clock()}] finished {algo} seed {seed} in {mins:.0f} min: {final[-1].strip() if final else ''}",
                      flush=True)
            else:
                print(f"[{_clock()}] FAILED {algo} seed {seed} (exit {p.returncode}); last lines:\n"
                      + "\n".join(open(log).read().splitlines()[-15:]), flush=True)
        if running and time.time() - last > report_every:
            last = time.time()
            print(f"[{_clock()}] progress:", flush=True)
            for (algo, seed), (p, log, t0) in running.items():
                print(f"    {_last_eval(log)}", flush=True)
        time.sleep(15)
    print(f"[{_clock()}] all trainings done", flush=True)

## 2. Training
MACURA and MBPO of every seed start together; SAC starts when a slot is free. A progress line per training appears
every 15 minutes (its latest evaluation); each training's full log is in `runs/joblogs/`. About 4-5 h (estimate).

In [ ]:
JOBS = [(a, s) for s in SEEDS for a in TOGETHER] + [(a, s) for s in SEEDS for a in AFTER]
print("SEEDS", SEEDS, "->", len(JOBS), "trainings:", ", ".join(f"{a} seed {s}" for a, s in JOBS))
run_jobs(JOBS, N_JOBS)

## 3. Results of this session
- **Per seed**: the scores fixed before the run (average return while learning, drones broken after the warm-up),
  then the best checkpoint's test on 30 fresh scenarios.
- **MACURA - MBPO per seed** (same seed = same scenarios, wind and gusts), and in how many seeds MACURA is ahead.
- **Updates per real step**: proof that MACURA and MBPO trained about equally.
- **The chute**: did a *trained* policy lose lift? Only the second half of training and the final test count, so
  beginners that fall do not.

In [ ]:
RUNS = R.load_runs(OUT)
print("SEEDS in this session's results:", sorted({r["seed"] for r in RUNS}))
display(Markdown("### Per seed\n\n" + R.per_seed_markdown(RUNS)))

GAP = [("avg_return", "avg return while learning", "{:+.0f}", 1), ("broken", "drones broken after warm-up", "{:+.0f}", -1),
       ("test_return", "final test return", "{:+.0f}", 1), ("test_crash", "final test crash rate", "{:+.0%}", -1),
       ("test_success", "success, % of a lap (test)", "{:+.0%}", 1)]
by = {(r["algo"], r["seed"]): R.run_scores(r) for r in RUNS}
seeds = sorted({s for (a, s) in by if (("macura", s) in by and ("mbpo", s) in by)})
rows = ["| MACURA - MBPO | " + " | ".join(f"seed {s}" for s in seeds) + " | MACURA better in |",
        "|---|" + "---|" * (len(seeds) + 1)]
for k, label, fmt, sign in GAP:
    d = [by[("macura", s)][k] - by[("mbpo", s)][k] for s in seeds]
    rows.append(f"| {label} | " + " | ".join(fmt.format(x) for x in d)
                + f" | {sum(sign * x > 0 for x in d)} of {len(d)} seeds |")
display(Markdown("### MACURA - MBPO per seed\n\n" + "\n".join(rows)))

utd = ["| algorithm | seed | updates per real step (mean) |", "|---|---|---|"]
for r in RUNS:
    u = f"{np.mean(r['utd']):.1f}" if r.get("utd") else "1 (model-free)"
    utd.append(f"| {R.NAMES[r['algo']]} | {r['seed']} | {u} |")
display(Markdown("### Updates per real step\n\n" + "\n".join(utd)))

chute = ["| algorithm | seed | fastest descent, 2nd half (m/s) | time losing lift, 2nd half | test: fastest descent | "
         "test: time losing lift | MACURA trusts fast / normal |", "|---|---|---|---|---|---|---|"]
reached = []
for r in RUNS:
    st = np.asarray(r["steps"]); late = st >= STEPS / 2
    sink = np.asarray(r["eval_max_sink"])[late]; lift = np.asarray(r["eval_liftloss"])[late]
    fe = r["final_eval"]
    tr = "-"
    if r["algo"] == "macura" and r.get("trust_fast"):
        tr = (f"{np.nanmean([v for _, v in r['trust_fast']]):.0%} / "
              f"{np.nanmean([v for _, v in r['trust_slow']]):.0%}")
    if (len(lift) and lift.mean() > 0.01) or fe.get("eval_liftloss", 0) > 0.01:
        reached.append(f"{R.NAMES[r['algo']]} seed {r['seed']}")
    chute.append(f"| {R.NAMES[r['algo']]} | {r['seed']} | {sink.max() if len(sink) else float('nan'):.2f} | "
                 f"{100 * lift.mean() if len(lift) else float('nan'):.1f}% | {fe.get('eval_max_sink', float('nan')):.2f} | "
                 f"{100 * fe.get('eval_liftloss', 0):.1f}% | {tr} |")
display(Markdown("### The chute (lift loss starts at 1.2 m/s)\n\n" + "\n".join(chute)))
print("Trained policies losing lift more than 1% of the time: " + (", ".join(reached) if reached else
      "none - the chute trap did not come into play for trained policies"))

REFS = R.autopilot_reference(cfg.ENV, seed_base=100, episodes=20, cache=f"{OUT}/autopilot_reference.json")
F.plot_learning_curves(RUNS, REFS, f"{OUT}/plots/learning_curves.png"); plt.show()
F.plot_model_trust(RUNS, f"{OUT}/plots/model_trust.png"); plt.show()
open(f"{OUT}/summary.md", "w").write("\n\n".join([f"# RACE3 test, seeds {SEEDS}", R.per_seed_markdown(RUNS),
                                                   "\n".join(rows), "\n".join(utd), "\n".join(chute)]))
print("saved:", f"{OUT}/summary.md")

## 4. Outputs
`/kaggle/working/runs`: `logs/` (one JSON per algorithm and seed), `checkpoints/`, `joblogs/` (each training's
printout), `plots/`, `summary.md`. Download each account's output into its own folder (e.g. `out_race3_seeds01`,
`out_race3_seeds23`); the two are then merged on the computer for the decision (4 seeds).